# Project 4 — CNN Image Classifier

## Overview
We train a **Convolutional Neural Network (CNN)** on MNIST using PyTorch, reaching >99% accuracy. Unlike fully-connected networks, CNNs exploit the spatial structure of images through local connectivity and weight sharing.

**Architecture:**
```
Input(1×28×28) → Conv(32) → ReLU → Conv(64) → ReLU → MaxPool → Dropout
               → Flatten → Linear(128) → ReLU → Dropout → Linear(10) → Softmax
```

## 1. Theory — Convolutional Neural Networks

### Convolution Operation
A convolution slides a filter $\mathbf{K}$ (kernel) of size $k \times k$ over an input feature map $\mathbf{X}$:
$$(\mathbf{X} * \mathbf{K})_{i,j} = \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} \mathbf{X}_{i+m,\, j+n} \cdot \mathbf{K}_{m,n}$$

Each filter learns to detect a specific **pattern** (edges, curves, textures). Multiple filters produce multiple **feature maps**.

### Output Size
Given input $H \times W$, kernel $k$, padding $p$, stride $s$:
$$H_{out} = \left\lfloor \frac{H + 2p - k}{s} \right\rfloor + 1$$

### Max Pooling
Reduces spatial size by taking the maximum over a window:
$$\text{MaxPool}_{2\times2}(\mathbf{X})_{i,j} = \max\left(\mathbf{X}_{2i:2i+2,\, 2j:2j+2}\right)$$

This introduces **translation invariance** and reduces parameters.

### Batch Normalisation
Normalises layer inputs across the batch:
$$\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} \cdot \gamma + \beta$$

Allows higher learning rates and stabilises training.

### Dropout
Randomly zeroes units with probability $p$ during training:
$$y = \frac{1}{1-p} \cdot x \odot \text{Bernoulli}(1-p)$$

Acts as an ensemble regularisation technique — prevents co-adaptation of neurons.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

sns.set_theme(style='whitegrid')

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

## 2. Dataset Loading

In [ ]:
# MNIST-specific normalisation values (precomputed mean and std)
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,))
])

train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=train_transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=64,   shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=1000, shuffle=False, num_workers=2)

print(f'Training samples: {len(train_dataset):,}')
print(f'Test samples:     {len(test_dataset):,}')

# Visualise a batch
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze().numpy(), cmap='gray')
    ax.set_title(labels[i].item())
    ax.axis('off')
plt.suptitle('Training Batch Sample')
plt.tight_layout()
plt.show()

## 3. CNN Architecture

In [ ]:
class CNN(nn.Module):
    """
    CNN for MNIST classification.

    Architecture:
        Input: (B, 1, 28, 28)
        Conv1: (B, 32, 26, 26)  -- 3×3 kernel, no padding
        Conv2: (B, 64, 24, 24)  -- 3×3 kernel, no padding
        MaxPool: (B, 64, 12, 12)
        Dropout(0.25)
        Flatten: (B, 9216)
        Linear: (B, 128)
        Dropout(0.5)
        Linear: (B, 10)
    """

    def __init__(self):
        super().__init__()

        # Convolutional feature extractor
        self.conv1   = nn.Conv2d(in_channels=1,  out_channels=32, kernel_size=3)  # 28→26
        self.conv2   = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)  # 26→24
        self.pool    = nn.MaxPool2d(kernel_size=2)                                # 24→12
        self.drop1   = nn.Dropout(p=0.25)

        # Classifier head
        self.fc1     = nn.Linear(64 * 12 * 12, 128)
        self.drop2   = nn.Dropout(p=0.5)
        self.fc2     = nn.Linear(128, 10)

    def forward(self, x):
        # Feature extraction
        x = F.relu(self.conv1(x))   # conv + relu
        x = F.relu(self.conv2(x))   # conv + relu
        x = self.pool(x)            # spatial downsampling
        x = self.drop1(x)           # regularisation

        # Flatten spatial dims for FC layers
        x = x.view(x.size(0), -1)  # (B, 64*12*12)

        # Classification
        x = F.relu(self.fc1(x))
        x = self.drop2(x)
        x = self.fc2(x)             # raw logits (CrossEntropyLoss applies softmax)
        return x


model = CNN().to(device)
print(model)

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal trainable parameters: {total_params:,}')

## 4. Training

In [ ]:
optimizer   = optim.Adam(model.parameters(), lr=1e-3)
criterion   = nn.CrossEntropyLoss()
EPOCHS      = 10
train_losses, test_accuracies = [], []


def train_epoch(model, loader, optimizer, criterion):
    model.train()  # training mode (dropout active)
    total_loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()          # clear previous gradients
        outputs = model(images)        # forward pass
        loss    = criterion(outputs, labels)  # compute loss
        loss.backward()                # backpropagate
        optimizer.step()               # update weights
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()  # evaluation mode (dropout inactive)
    correct, total = 0, 0
    with torch.no_grad():  # no gradient computation needed
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs  = model(images)
            predicted = outputs.argmax(dim=1)
            correct  += (predicted == labels).sum().item()
            total    += labels.size(0)
    return correct / total


print(f'Training for {EPOCHS} epochs on {device}...')
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    test_acc   = evaluate(model, test_loader)

    train_losses.append(train_loss)
    test_accuracies.append(test_acc)

    print(f'Epoch {epoch:>2}/{EPOCHS}  Loss: {train_loss:.4f}  Test Acc: {test_acc:.4f}')

print(f'\nFinal test accuracy: {test_accuracies[-1]:.4f}')

In [ ]:
# Save model checkpoint
torch.save(model.state_dict(), 'mnist_cnn.pt')
print('Model checkpoint saved to mnist_cnn.pt')

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, EPOCHS+1), train_losses, 'o-', color='royalblue')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training Loss')

axes[1].plot(range(1, EPOCHS+1), test_accuracies, 'o-', color='darkorange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Test Accuracy')
axes[1].set_ylim([0.9, 1.0])

plt.tight_layout()
plt.show()

## 5. Detailed Evaluation

In [ ]:
# Collect all predictions
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds   = outputs.argmax(dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# Per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1)
print('\nPer-class accuracy:')
for digit, acc in enumerate(per_class_acc):
    print(f'  Digit {digit}: {acc:.4f}')

In [ ]:
# Visualise predictions with confidence
model.eval()
sample_images, sample_labels = next(iter(test_loader))
sample_images = sample_images[:10]
sample_labels = sample_labels[:10]

with torch.no_grad():
    logits = model(sample_images.to(device))
    probs  = F.softmax(logits, dim=1).cpu().numpy()
    preds  = logits.argmax(dim=1).cpu().numpy()

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(sample_images[i].squeeze().numpy(), cmap='gray')
    pred_label = preds[i]
    true_label = sample_labels[i].item()
    confidence = probs[i][pred_label]
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(f'Pred: {pred_label} ({confidence:.2%})\nTrue: {true_label}', color=color)
    ax.axis('off')
plt.suptitle('Predictions with Confidence')
plt.tight_layout()
plt.show()

## Summary

| Component | Details |
|---|---|
| Architecture | 2 Conv layers + 2 FC layers |
| Parameters | ~1.2M trainable |
| Optimiser | Adam (lr=1e-3) |
| Regularisation | Dropout (0.25, 0.5) |
| Target | >99% test accuracy on MNIST |

**Key insight**: CNNs vastly outperform MLPs on image data because convolutional filters share weights across spatial positions, allowing the network to detect features regardless of location (translation equivariance).